# POLKORP AI — Kaggle GPU Notebook

Opening the world of AI for you.

Runs Ollama + Lama Cleaner + Fooocus on Kaggle's free T4 GPU (30h/week quota).
Each service is exposed via its own public URL — no ngrok required, Fooocus
uses Gradio's built-in `--share` tunnel and Lama Cleaner is reached through
Kaggle's port-forwarding proxy.

**Interactive use (browser):** Settings (right sidebar) → Accelerator → GPU T4 x2,
and Internet must be turned on, then Run All. The session stays alive as long
as the browser tab is open (up to Kaggle's ~9-12h session cap).

**API/unattended use (`kaggle kernels push`):** batch-pushed kernels have no
browser tab keeping them open — the session ends the moment the last cell
finishes. The final "Keep alive" cell below holds the session open for a
fixed number of hours so the background services stay reachable either way.

A Simple Corp.

## 1. Install Ollama and pull a base model

In [ ]:
# zstd is required by the ollama installer to extract its archive —
# not preinstalled in Kaggle's default image.
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
try:
    ollama_proc = subprocess.Popen(["ollama", "serve"])
    time.sleep(5)
    print("Ollama server started (pid:", ollama_proc.pid, ")")
except Exception as e:
    print("Ollama failed to start —", e, "— continuing to the next service.")


In [ ]:
!ollama pull llama3.1:8b

## 2. Install and launch Lama Cleaner (port 8080)

In [ ]:
# lama-cleaner pulls in `tokenizers`, which needs a Rust compiler to
# build from source on Kaggle's current Python version (no prebuilt
# wheel available yet). Install Rust first so pip can build it.
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

!pip install -q lama-cleaner

import subprocess
try:
    lama_proc = subprocess.Popen(
        ["lama-cleaner", "--device", "cuda", "--port", "8080", "--host", "0.0.0.0"]
    )
    print("Lama Cleaner starting on port 8080 (pid:", lama_proc.pid, ")")
    print("Open it via Kaggle's Output tab -> port 8080, once it's ready (~30s).")
except Exception as e:
    print("Lama Cleaner failed to start —", e, "— continuing to the next service.")


## 3. Clone and launch Fooocus (port 7865, public Gradio share link)

In [ ]:
!git clone --depth 1 https://github.com/lllyasviel/Fooocus.git /kaggle/working/Fooocus
%cd /kaggle/working/Fooocus
!pip install -q -r requirements_versions.txt


In [ ]:
# --share gives a public Gradio tunnel URL printed in the cell output below —
# that's the "no ngrok needed" link for Fooocus.
import subprocess
try:
    fooocus_proc = subprocess.Popen(
        ["python3", "entry_with_update.py", "--share", "--listen", "--port", "7865"],
        cwd="/kaggle/working/Fooocus"
    )
    print("Fooocus launching — watch this cell's output for the public *.gradio.live URL.")
except Exception as e:
    print("Fooocus failed to start —", e, "— continuing.")


## 4. Summary of running services

In [ ]:
print("POLKORP AI — Kaggle GPU session")
print("--------------------------------")
print("Ollama (llama3.1:8b):  http://localhost:11434  (use Kaggle's port-forward proxy)")
print("Lama Cleaner:          http://localhost:8080    (use Kaggle's port-forward proxy)")
print("Fooocus:               see the *.gradio.live URL printed by the cell above")
print("")
print("Remember: Kaggle GPU sessions have a 30h/week quota and a ~9-12h max runtime.")
print("A Simple Corp.")

## 5. Keep alive

Holds the kernel open so Ollama/Lama Cleaner/Fooocus stay reachable. Needed
for API/unattended runs (no browser tab keeping the session open); harmless
to leave running in interactive mode too — just stop this cell whenever
you're done and want to free the GPU quota early.

In [ ]:
import time

HOURS_TO_STAY_ALIVE = 8  # stay under Kaggle's ~9-12h session cap
print(f"Keeping this kernel alive for up to {HOURS_TO_STAY_ALIVE}h so the "
      "services above stay reachable. Stop this cell any time to end early.")

end_time = time.time() + HOURS_TO_STAY_ALIVE * 3600
beat = 0
while time.time() < end_time:
    time.sleep(300)
    beat += 1
    remaining_min = int((end_time - time.time()) / 60)
    print(f"[heartbeat {beat}] still alive — {remaining_min} min remaining")
